# ConceptTester — Fine-tune MULTI-SEED + teacher API (Kaggle GPU)

3 điều kiện: **(1) LLM thường · (2) Baseline (target mù) · (3) Concept (target có hướng)**.
So sánh chính = **ATTACK / ATK_cln** của (3) vs (2), báo **mean ± std qua nhiều seed**.

**Mới so bản trước:**
- **Multi-seed**: train+infer lặp qua `SEEDS` → mean±std (chống nhiễu thống kê).
- **Teacher qua API**: `TEACHER="groq"` (ChatGroq) — mạnh hơn 3B local, gần như free (30 RPM / 1000 req-ngày).
- **Tăng data distill**: `N_TRAIN=350, N_TEST=80`.
- **Prompt-fix**: concept-target bắt buộc **gọi hàm gốc** + **oracle crash (EXPECT_DEATH/ASAN)** → vá `calls_focal`/`crash_orc`.
- **Metric baked-in**: has_test, ATTACK, **ATK_cln** (bỏ copy), **copy**, **calls_focal**, **crash_orc**, **line_hit**, rationale.

### Chuẩn bị
1. **Accelerator ▸ GPU**. 2. **Add Data**: `train_concept.jsonl` (file to). 3. **Secrets ▸ `GROQ_API_KEY`** (lấy free ở console.groq.com).
> Distill dùng cache (sha1) → chỉ gọi API 1 lần/hàm, chia sẻ qua mọi seed. ~430 hàm × 2 = ~860 call, vừa hạn mức 1000/ngày của Groq free.
> Thời lượng ~1.5–2h (3 seed). Muốn nhanh: đặt `SEEDS="42"` (1 seed).


In [ ]:
# === Cell 1: Dependencies ===
import subprocess, sys
def _pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=False)
_pip("-U","bitsandbytes","peft","accelerate")
_pip("langchain-groq","langchain-google-genai")
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!"))

In [ ]:
# === Cell 2: Cấu hình ===
import os, glob, json, re, random
BASE_MODEL    = os.environ.get("BASE_MODEL", "Qwen/Qwen2.5-Coder-1.5B-Instruct")   # student
TEACHER       = os.environ.get("TEACHER", "groq")                                  # groq | local | gemini
TEACHER_MODEL = os.environ.get("TEACHER_MODEL", "Qwen/Qwen2.5-Coder-7B-Instruct")  # khi local
GROQ_MODEL    = os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile")            # hoac: qwen3-32b, openai/gpt-oss-120b
GROQ_RPM      = int(os.environ.get("GROQ_RPM", "30"))                               # free tier 30/phut
GEMINI_MODEL  = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
SEEDS   = [int(x) for x in os.environ.get("SEEDS","42,123").split(",")]   # 2 seed cho an toan thoi gian; them "7" neu con gio
N_TRAIN = int(os.environ.get("N_TRAIN", "350"))
N_TEST  = int(os.environ.get("N_TEST", "80"))
MAX_LEN, EPOCHS, POOL_SEED = 1024, 3, 42
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

def _secret(name):
    v=os.environ.get(name)
    if v: return v
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception: return None
GROQ_API_KEY   = _secret("GROQ_API_KEY")
GEMINI_API_KEY = _secret("GEMINI_API_KEY") or _secret("GOOGLE_API_KEY")

print("student:", BASE_MODEL)
print("teacher:", TEACHER, "|", {"groq":GROQ_MODEL,"local":TEACHER_MODEL,"gemini":GEMINI_MODEL}.get(TEACHER))
print("SEEDS:", SEEDS, "| N_TRAIN/N_TEST:", N_TRAIN, N_TEST)
if TEACHER=="groq":  print("GROQ_API_KEY:", "FOUND" if GROQ_API_KEY else "THIEU -> them Secret GROQ_API_KEY")
if TEACHER=="gemini":print("GEMINI key:", "FOUND" if GEMINI_API_KEY else "THIEU")

In [ ]:
# === Cell 3: Nạp pool data (lọc CWE thật + concept nhất quán) ===
PROPS=["null_assignment","null_check","ptr_deref","buffer_allocation","buffer_access","bound_check","free","if_statement"]
CWE_FAMILY={"CWE-787":"oob_write","CWE-125":"oob_read","CWE-119":"oob","CWE-120":"oob_write","CWE-131":"oob",
            "CWE-416":"uaf","CWE-415":"double_free","CWE-476":"null_deref","CWE-190":"int_overflow","CWE-369":"div_zero"}
NEED={"oob":[4,3],"oob_write":[4,3],"oob_read":[4,3],"uaf":[6],"double_free":[6],"null_deref":[2],"int_overflow":[],"div_zero":[]}
def _norm_cwe(c):
    if isinstance(c,list): return [x for x in c if isinstance(x,str) and re.match(r"CWE-\d+$",x)]
    if isinstance(c,str) and re.match(r"CWE-\d+$",c): return [c]
    return []
def _find_vd():
    for p in ["/kaggle/input/**/train_concept.jsonl","/kaggle/input/**/teest_concept.jsonl",
              "/kaggle/input/**/*concept*.jsonl","data/VD/train_concept.jsonl"]:
        h=glob.glob(p, recursive=True)
        if h: return h[0]
    return None
def load_pool(n):
    path=_find_vd(); rows=[]
    if not path: print("[warn] khong thay VD jsonl - Add Data!"); return []
    print("VD file:", path)
    for line in open(path):
        d=json.loads(line)
        if d.get("target")!=1 or not d.get("concepts"): continue
        if not (6<=len(d["func"].splitlines())<=28): continue
        cwes=_norm_cwe(d.get("cwe")); fam=next((CWE_FAMILY[c] for c in cwes if c in CWE_FAMILY),None)
        if not fam: continue
        need=NEED.get(fam,[])
        if need and not any(d["concepts"][i] for i in need): continue
        d["_cwe"]=next(c for c in cwes if c in CWE_FAMILY); d["_family"]=fam; rows.append(d)
    random.Random(POOL_SEED).shuffle(rows)
    return rows[:n]
ALL=load_pool(N_TRAIN+N_TEST)
import collections
print("pool:", len(ALL), "| ho:", dict(collections.Counter(d["_family"] for d in ALL)))

In [ ]:
# === Cell 4: Recipe tấn công + focal + build message (prompt-fix) ===
FAMILY_DESC={"oob_write":"GHI ngoai bien","oob_read":"DOC ngoai bien","oob":"truy cap ngoai bien buffer",
 "uaf":"dung sau khi free (use-after-free)","double_free":"free hai lan","null_deref":"deref con tro NULL",
 "int_overflow":"tran so nguyen khi tinh kich thuoc","div_zero":"chia cho khong"}
RECIPE={"oob_write":"truyen do dai/chi so LON HON kich thuoc buffer (n > sizeof) de GHI vuot bien.",
 "oob_read":"chi so/do dai vuot vung hop le de DOC vuot bien.","oob":"input vuot kich thuoc buffer.",
 "uaf":"sau khi con tro da free, GOI lai duong dan DUNG con tro do.","double_free":"dan toi free CUNG con tro HAI LAN.",
 "null_deref":"TRUYEN NULL vao tham so con tro roi goi ham.","int_overflow":"gia tri gan INT_MAX/SIZE_MAX gay TRAN.",
 "div_zero":"mau so = 0."}
def focal(func):
    m=re.search(r'([A-Za-z_]\w*)\s*\([^{;]*\)\s*(?:/\*[\s\S]*?\*/\s*)?\{',func)
    return m.group(1) if m else None
def concept_lines(d):
    out=[]
    for i,ln in enumerate(d["func"].split("\n")):
        vec=d["lines_concept"].get(str(i)) or [0]*8
        tags=[PROPS[j] for j,v in enumerate(vec) if v]
        if tags: out.append("  - dong %2d [%s] -> %s"%(i,", ".join(tags),ln.strip()))
    return "\n".join(out)
SYS_BLIND="Ban la ky su kiem thu C/C++. Sinh unit test cho ham. Chi tra code, comment ngan."
SYS_CONCEPT=("Ban la ky su kiem thu bao mat C/C++. Sinh unit test NHAM DUNG lo hong, GOI ham goc voi input doc hai, "
             "va co ORACLE bao loi (EXPECT_DEATH/ASSERT_DEATH hoac ky vong AddressSanitizer). Chi tra code, comment ngan.")
def build_messages_blind(d):
    return [{"role":"system","content":SYS_BLIND},
            {"role":"user","content":"Ham can kiem thu:\n```c\n%s\n```"%d["func"]}]
def build_messages_concept(d):
    fam=d["_family"]; fn=focal(d["func"]) or "ham_can_test"
    user=("Ham can kiem thu:\n```c\n%s\n```\n\n"
          "LO HONG (phan tich tinh): %s = %s.\nCac dong thao tac lien quan:\n%s\n\n"
          "YEU CAU:\n- %s\n- Test PHAI GOI ham `%s(...)` voi input doc hai o tren.\n"
          "- PHAI co oracle: EXPECT_DEATH/ASSERT_DEATH hoac chu thich ky vong AddressSanitizer bao loi.\n"
          "- Moi test bat dau bang: // TARGET: %s @ L<dong> -> <ky thuat>"
          %(d["func"], d["_cwe"], FAMILY_DESC.get(fam,fam), concept_lines(d), RECIPE.get(fam,""), fn, d["_cwe"]))
    return [{"role":"system","content":SYS_CONCEPT},{"role":"user","content":user}]
print("messages ready | vi du focal:", focal(ALL[0]["func"]) if ALL else "N/A")

In [ ]:
# === Cell 5: DUAL distill (teacher=groq|local|gemini) — cache sha1, chia se qua seed ===
import hashlib, time
CACHE=os.path.join(WORK,"distill_cache_v21.jsonl"); _cache={}
# doc cache tu WORK + moi distill_cache*.jsonl trong /kaggle/input
# -> neu rerun, Add Data output cu (co file cache) thi KHONG phai distill lai
_cfiles=([CACHE] if os.path.exists(CACHE) else [])+glob.glob("/kaggle/input/**/distill_cache*.jsonl",recursive=True)
for cf in _cfiles:
    for l in open(cf):
        try: o=json.loads(l); _cache[o["key"]]=o["target"]
        except Exception: pass
print("cache:", len(_cache), "tu", len(_cfiles), "file")
def _key(func,cond): return hashlib.sha1((cond+"::"+func).encode()).hexdigest()
def _save(k,o): _cache[k]=o; open(CACHE,"a").write(json.dumps({"key":k,"target":o})+"\n")

def _api_caller():
    if TEACHER=="groq":
        from langchain_groq import ChatGroq
        from langchain_core.messages import SystemMessage, HumanMessage
        llm=ChatGroq(model=GROQ_MODEL, temperature=0.3, groq_api_key=GROQ_API_KEY); rpm=GROQ_RPM
    else:
        from langchain_google_genai import ChatGoogleGenerativeAI
        from langchain_core.messages import SystemMessage, HumanMessage
        llm=ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0.3, google_api_key=GEMINI_API_KEY); rpm=int(os.environ.get("GEMINI_RPM","10"))
    mi=60.0/max(1,rpm); last=[0.0]
    def call(msgs):
        for a in range(5):
            dt=time.monotonic()-last[0]
            if dt<mi: time.sleep(mi-dt)
            last[0]=time.monotonic()
            try: return llm.invoke([SystemMessage(content=msgs[0]["content"]),HumanMessage(content=msgs[1]["content"])]).content
            except Exception as e:
                w=min(60,4*2**a); print("[warn] retry %ds: %s"%(w,str(e)[:90])); time.sleep(w)
        return None
    return call

def distill_local_gen():
    import torch,gc
    from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
    tk=AutoTokenizer.from_pretrained(TEACHER_MODEL)
    if tk.pad_token is None: tk.pad_token=tk.eos_token
    print("Nap teacher local:",TEACHER_MODEL); tm=AutoModelForCausalLM.from_pretrained(TEACHER_MODEL,quantization_config=bnb,device_map="auto"); tm.eval()
    def call(msgs):
        enc=tk.apply_chat_template(msgs,add_generation_prompt=True,return_tensors="pt")
        enc={k:v.to(tm.device) for k,v in enc.items()} if hasattr(enc,"keys") else {"input_ids":enc.to(tm.device)}
        with torch.no_grad(): o=tm.generate(**enc,max_new_tokens=512,do_sample=True,temperature=0.3,top_p=0.9,pad_token_id=tk.pad_token_id)
        return tk.decode(o[0][enc["input_ids"].shape[1]:],skip_special_tokens=True)
    return call, (lambda: (tm.__delattr__ if False else None) or __import__("gc").collect() or torch.cuda.empty_cache())

# LAZY: chi dung teacher (ChatGroq/local) khi co cache-miss thuc su
# -> neu cache day (distill da chay LOCAL + Add Data), Kaggle KHONG can key/teacher
_caller=[None]; _cleanup=[None]
def get_call():
    if _caller[0] is None:
        if TEACHER=="local": _caller[0], _cleanup[0] = distill_local_gen()
        else:                _caller[0] = _api_caller()
    return _caller[0]

kept=[]; n_api=0
for i,d in enumerate(ALL):
    for cond,mk in (("blind",build_messages_blind),("concept",build_messages_concept)):
        k=_key(d["func"],cond)
        if k in _cache: d["tgt_"+cond]=_cache[k]; continue
        o=get_call()(mk(d)); n_api+=1
        if o and len(o)>40: _save(k,o); d["tgt_"+cond]=o
    if d.get("tgt_blind") and d.get("tgt_concept"): kept.append(d)
    if (i+1)%20==0: print("  %d/%d (kept %d, api-call %d)"%(i+1,len(ALL),len(kept),n_api))
if _cleanup[0]: _cleanup[0]()
ALL=kept
print("Sau distill:", len(ALL), "| goi API:", n_api, "(0 = dung cache hoan toan, khong can key)")

In [ ]:
# === Cell 6: Tokenizer + loader 4-bit ===
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
_bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
def load_base():
    m=AutoModelForCausalLM.from_pretrained(BASE_MODEL,quantization_config=_bnb,device_map="auto"); m.config.use_cache=False; return m
def free_mem(*o):
    for x in o:
        try: del x
        except Exception: pass
    gc.collect(); torch.cuda.empty_cache()
print("tokenizer ready:", BASE_MODEL)

In [ ]:
# === Cell 7: Tokenize + collator ===
def tokenize_pair(msgs,target):
    pt=tokenizer.apply_chat_template(msgs,add_generation_prompt=True,tokenize=False)
    ft=tokenizer.apply_chat_template(msgs+[{"role":"assistant","content":target}],add_generation_prompt=False,tokenize=False)
    pid=tokenizer(pt,add_special_tokens=False)["input_ids"]; fid=tokenizer(ft,add_special_tokens=False)["input_ids"]
    lab=[-100]*len(pid)+fid[len(pid):]
    return {"input_ids":[int(x) for x in fid[:MAX_LEN]],"labels":[int(x) for x in lab[:MAX_LEN]],"attention_mask":[1]*len(fid[:MAX_LEN])}
class PadCollator:
    def __init__(s,p): s.p=p
    def __call__(s,b):
        L=max(len(x["input_ids"]) for x in b); o={"input_ids":[],"labels":[],"attention_mask":[]}
        for x in b:
            d=L-len(x["input_ids"]); o["input_ids"].append(x["input_ids"]+[s.p]*d); o["labels"].append(x["labels"]+[-100]*d); o["attention_mask"].append(x["attention_mask"]+[0]*d)
        return {k:torch.tensor(v) for k,v in o.items()}
collator=PadCollator(tokenizer.pad_token_id)
print("tokenize ready")

In [ ]:
# === Cell 8: split(seed) + train + infer + run_seed ===
from collections import defaultdict
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from transformers import Trainer, TrainingArguments
class _DS(torch.utils.data.Dataset):
    def __init__(s,d): s.d=d
    def __len__(s): return len(s.d)
    def __getitem__(s,i): return s.d[i]

def split(seed):
    byp=defaultdict(list)
    for d in ALL: byp[d.get("project","?")].append(d)
    projs=[p for p in byp if p!="linux"]; random.Random(seed).shuffle(projs)
    tt=min(N_TEST,max(5,int(0.25*len(ALL)))); test=[]
    for p in projs:
        if len(test)>=tt: break
        test.extend(byp[p])
    tid={id(x) for x in test}; train=[d for d in ALL if id(d) not in tid][:N_TRAIN]
    return train,test

def train_adapter(rows,cond,out_dir,seed):
    data=[tokenize_pair(build_messages_concept(d) if cond=="concept" else build_messages_blind(d),
                        d["tgt_concept"] if cond=="concept" else d["tgt_blind"]) for d in rows if d.get("tgt_"+cond)]
    model=prepare_model_for_kbit_training(load_base())
    lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias="none",task_type="CAUSAL_LM",
                    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
    model=get_peft_model(model,lora)
    args=TrainingArguments(output_dir=out_dir,num_train_epochs=EPOCHS,per_device_train_batch_size=1,gradient_accumulation_steps=8,
        learning_rate=2e-4,lr_scheduler_type="cosine",warmup_ratio=0.03,logging_steps=25,save_strategy="no",bf16=True,
        optim="paged_adamw_8bit",report_to="none",seed=seed)
    Trainer(model=model,args=args,train_dataset=_DS(data),data_collator=collator).train()
    model.save_pretrained(out_dir); free_mem(model)

@torch.no_grad()
def gen(model,msgs,mx=384):
    enc=tokenizer.apply_chat_template(msgs,add_generation_prompt=True,return_tensors="pt")
    enc={k:v.to(model.device) for k,v in enc.items()} if hasattr(enc,"keys") else {"input_ids":enc.to(model.device)}
    o=model.generate(**enc,max_new_tokens=mx,do_sample=False,use_cache=True,pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(o[0][enc["input_ids"].shape[1]:],skip_special_tokens=True)

def run_cond(adapter,concept_input,test_rows):
    base=load_base(); model=PeftModel.from_pretrained(base,adapter) if adapter else base
    base.config.use_cache=True; model.config.use_cache=True   # BAT KV-cache cho infer (khong -> cham ~10x)
    model.eval()
    res=[]
    for d in test_rows:
        msgs=build_messages_concept(d) if concept_input else build_messages_blind(d)
        res.append({"func":d["func"],"cwe":d["_cwe"],"family":d["_family"],"lines_concept":d["lines_concept"],"output":gen(model,msgs)})
    free_mem(model,base); return res

def run_seed(seed):
    tr,te=split(seed); print("  seed %d: train=%d test=%d"%(seed,len(tr),len(te)))
    dB=os.path.join(WORK,"adB_%d"%seed); dC=os.path.join(WORK,"adC_%d"%seed)
    train_adapter(tr,"blind",dB,seed); train_adapter(tr,"concept",dC,seed)
    return {"1_llm":run_cond(None,False,te),"2_baseline":run_cond(dB,False,te),"3_concept":run_cond(dC,True,te)}
print("run_seed ready")

In [ ]:
# === Cell 9: Chạy qua NHIỀU SEED ===
ALL_RESULTS={}
if ALL:
    for s in SEEDS:
        print("===== SEED %d ====="%s); ALL_RESULTS[str(s)]=run_seed(s)
    json.dump(ALL_RESULTS, open(os.path.join(WORK,"results_multiseed.json"),"w"))
    print("saved results_multiseed.json | seeds:", list(ALL_RESULTS))
else:
    print("[skip] khong co data")

In [ ]:
# === Cell 10: Metrics mean±std qua seed (v3 attack_clean/copy + v5 calls_focal/crash/line_hit) ===
import statistics as st, re
_ST={"if","else","for","while","return","int","void","struct","const","static","sizeof","NULL","nullptr","char"}
def _tk(s): return set(re.findall(r"[A-Za-z_]\w*",s))-_ST
ATTACK={
 "oob_write": r'(out.?of.?bound|buffer.?overflow|overflow|beyond|too\s*(big|large|long)|larger\s+than|exceed|oversiz|"[A-Za-z0-9_]{16,}"|memcpy\s*\([^;]*,\s*\d{3,}\s*\)|str(n?cpy|cat)\s*\([^;]*"[A-Za-z0-9_ ]{12,}"|>=?\s*\w*(len|size|count|max)\b)',
 "oob_read": r'(out.?of.?bound|read\s+beyond|beyond|overflow|index\s*>=?\s*\w|\[[^\]]*\+\s*\d+\s*\]|oversiz)',
 "oob": r'(out.?of.?bound|buffer.?overflow|overflow|beyond|too\s*(big|large|long)|larger\s+than|exceed|oversiz|"[A-Za-z0-9_]{16,}")',
 "uaf": r'(use.?after.?free|dangling|free\s*\([^)]*\)[\s\S]{0,200}?(\*\s*\w|->|\[\s*\w))',
 "double_free": r'(double.?free|free\s*\([^;]*\)[\s\S]{0,200}?free\s*\()',
 "null_deref": r'(null.?pointer\s*deref|deref\w*\s*NULL|pass\w*\s*NULL|\b\w+\s*\([^)]*\bNULL\b[^)]*\))',
 "int_overflow": r'(INT_MAX|UINT_MAX|SIZE_MAX|LONG_MAX|2147483647|4294967295|0x[fF]{7,}|integer\s*overflow|wrap.?around)',
 "div_zero": r'(/\s*0\b|divid\w*\s*(by|cho)\s*(zero|0)|denominator\s*=\s*0)'}
NEED={"oob":[4,3],"oob_write":[4,3],"oob_read":[4,3],"uaf":[6],"double_free":[6],"null_deref":[2],"int_overflow":[],"div_zero":[]}
def ah(f,o): p=ATTACK.get(f); return 1 if (p and re.search(p,o or "",re.I)) else 0
def _nl(s): return [re.sub(r"\s+"," ",x).strip() for x in s.split("\n") if len(x.strip())>3]
def copyr(o,fn): fl=set(_nl(fn)); return len(fl&set(_nl(o)))/len(fl) if fl else 0.0
def minus(o,fn): fl=set(_nl(fn)); return "\n".join(x for x in o.split("\n") if re.sub(r"\s+"," ",x).strip() not in fl)
def _focal(func):
    m=re.search(r'([A-Za-z_]\w*)\s*\([^{;]*\)\s*(?:/\*[\s\S]*?\*/\s*)?\{',func); return m.group(1) if m else None
_HT=r"\bTEST\s*\(|\bTEST_F\s*\(|void\s+test\w*|int\s+main\s*\(|\bassert\b|ASSERT|EXPECT_|CHECK\(|REQUIRE\("
def score(r):
    o=r["output"] or ""; b=re.sub(r"(?m)^\s*//\s*TARGET.*$","",o); t=minus(b,r["func"]); fn=_focal(r["func"])
    m=re.search(r'@\s*L\s*(\d+)',o); lh=None
    if m and NEED.get(r["family"]):
        vec=r["lines_concept"].get(m.group(1)) or [0]*8; lh=1 if any(vec[i] for i in NEED[r["family"]]) else 0
    return {"has_test":1 if re.search(_HT,o,re.I) else 0,"attack":ah(r["family"],b),"attack_clean":ah(r["family"],t),
            "copy":round(copyr(o,r["func"]),3),
            "calls_focal":1 if (fn and re.search(r'\b'+re.escape(fn)+r'\s*\(',o)) else 0,
            "crash_orc":1 if re.search(r'EXPECT_DEATH|ASSERT_DEATH|ASSERT_EXIT|SIGSEGV|SIGABRT|\babort\b|sanitiz|ASAN|should\s*(crash|abort)',o,re.I) else 0,
            "rationale":1 if re.search(r"//\s*TARGET",o) else 0, "_line_hit":lh}
KEYS=["has_test","attack","attack_clean","copy","calls_focal","crash_orc","rationale"]
def cond_means(recs):
    S=[score(r) for r in recs]; out={k:st.mean([s[k] for s in S]) for k in KEYS}
    lh=[s["_line_hit"] for s in S if s["_line_hit"] is not None]; out["line_hit"]=(st.mean(lh) if lh else None)
    return out

if 'ALL_RESULTS' in dir() and ALL_RESULTS:
    seeds=list(ALL_RESULTS); nm={"1_llm":"LLM","2_baseline":"Baseline","3_concept":"Concept"}
    per={c:[cond_means(ALL_RESULTS[s][c]) for s in seeds] for c in ["1_llm","2_baseline","3_concept"]}
    print("seeds:",seeds,"  (mean ± std qua %d seed)\n"%len(seeds))
    hdr=["has_test","attack","ATK_cln","copy","calls_foc","crash","ration","line_hit"]
    print("%-9s "%""+" ".join("%11s"%h for h in hdr))
    mk={"has_test":"has_test","attack":"attack","ATK_cln":"attack_clean","copy":"copy","calls_foc":"calls_focal","crash":"crash_orc","ration":"rationale","line_hit":"line_hit"}
    for c in ["1_llm","2_baseline","3_concept"]:
        cells=[]
        for h in hdr:
            vals=[p[mk[h]] for p in per[c] if p.get(mk[h]) is not None]
            if not vals: cells.append("%11s"%"n/a"); continue
            m=st.mean(vals); sd=st.pstdev(vals) if len(vals)>1 else 0
            cells.append("%11s"%("%.2f±%.2f"%(m,sd)))
        print("%-9s "%nm[c]+" ".join(cells))
    print("\nSo sanh chinh: (3)Concept vs (2)Baseline o ATK_cln. calls_foc/crash = tac dung prompt-fix.")
else:
    print("[skip]")

In [ ]:
# === Cell 11: Ví dụ (seed đầu, 1 hàm, 3 điều kiện) ===
if 'ALL_RESULTS' in dir() and ALL_RESULTS:
    s0=list(ALL_RESULTS)[0]; R=ALL_RESULTS[s0]
    d=R["3_concept"][0]; print("SEED %s | HAM (%s/%s):\n%s\n"%(s0,d["cwe"],d["family"],d["func"]))
    for c in ["1_llm","2_baseline","3_concept"]:
        print("="*66,"\n",c,"\n","="*66); print((R[c][0]["output"] or "")[:1100])
else: print("[skip]")